# =============================================================
# SIMULADOR HÍBRIDO — 05
# =============================================================
### Usa resultados reais dos jogos já disputados do Mundial 2026 para construir a tabela de pontos da fase de grupos.

### Prevê apenas os jogos futuros com o modelo.

### LIMITAÇÃO: features de forma contaminadas pelos resultados reais do Mundial — data leakage alto.

### Para previsão genuína usar: 05b_simulator_frozen.ipynb
# =============================================================

In [ ]:
import pandas as pd
import numpy as np
import joblib

In [ ]:

df_full = pd.read_csv("../data/df_full.csv")
df_full["date"] = pd.to_datetime(df_full["date"])

wc_future = df_full[
    (df_full["tournament"] == "FIFA World Cup") &
    (df_full["home_score"].isna())
]

print(f"Jogos futuros do Mundial: {len(wc_future)}")
print(wc_future[["date", "home_team", "away_team"]].head(60).to_string())

Jogos futuros do Mundial: 44
            date               home_team       away_team
49775 2026-06-19                Scotland         Morocco
49776 2026-06-19                  Brazil           Haiti
49777 2026-06-19           United States       Australia
49778 2026-06-19                  Turkey        Paraguay
49779 2026-06-20                 Germany     Ivory Coast
49780 2026-06-20                 Ecuador         Curaçao
49781 2026-06-20             Netherlands          Sweden
49782 2026-06-20                 Tunisia           Japan
49783 2026-06-21                 Belgium            Iran
49784 2026-06-21             New Zealand           Egypt
49785 2026-06-21                   Spain    Saudi Arabia
49786 2026-06-21                 Uruguay      Cape Verde
49787 2026-06-22                  France            Iraq
49788 2026-06-22                  Norway         Senegal
49789 2026-06-22               Argentina         Austria
49790 2026-06-22                  Jordan         Algeria
49

In [25]:
wc_2026 = df_full[
    (df_full["tournament"] == "FIFA World Cup") &
    (df_full["date"] >= "2026-06-11")
]

teams_2026 = sorted(set(wc_2026["home_team"].tolist() + wc_2026["away_team"].tolist()))
print(f"Total de equipas no Mundial 2026: {len(teams_2026)}")
print(teams_2026)

Total de equipas no Mundial 2026: 48
['Algeria', 'Argentina', 'Australia', 'Austria', 'Belgium', 'Bosnia and Herzegovina', 'Brazil', 'Canada', 'Cape Verde', 'Colombia', 'Croatia', 'Curaçao', 'Czech Republic', 'DR Congo', 'Ecuador', 'Egypt', 'England', 'France', 'Germany', 'Ghana', 'Haiti', 'Iran', 'Iraq', 'Ivory Coast', 'Japan', 'Jordan', 'Mexico', 'Morocco', 'Netherlands', 'New Zealand', 'Norway', 'Panama', 'Paraguay', 'Portugal', 'Qatar', 'Saudi Arabia', 'Scotland', 'Senegal', 'South Africa', 'South Korea', 'Spain', 'Sweden', 'Switzerland', 'Tunisia', 'Turkey', 'United States', 'Uruguay', 'Uzbekistan']


In [ ]:
# --- 1. Carregar dados e modelos ---
df_full = pd.read_csv("../data/df_full.csv")
df_full["date"] = pd.to_datetime(df_full["date"])

clf_lr = joblib.load("../models/model_v3_clf_lr.pkl")  # para .predict()
clf_rf = joblib.load("../models/model_v3_clf_rf.pkl")  # para .predict_proba()

class EnsembleClassifier:
    def __init__(self, clf_predict, clf_proba):
        self.clf_predict = clf_predict
        self.clf_proba   = clf_proba
        self.classes_    = clf_proba.classes_

    def predict(self, X):
        return self.clf_predict.predict(X)

    def predict_proba(self, X):
        return self.clf_proba.predict_proba(X)

clf = EnsembleClassifier(clf_lr, clf_rf)

FEATURES = [
    "Diff_Ranking", "Fator_Casa",
    "Forma_Golos_Home", "Forma_Pts_Home", "Forma_Golos_Sofridos_Home",
    "Forma_Golos_Away", "Forma_Pts_Away", "Forma_Golos_Sofridos_Away",
    "Tipo_Competicao",
]

GOLOS_CAP = 5.0
for col in ["Forma_Golos_Home", "Forma_Golos_Away",
            "Forma_Golos_Sofridos_Home", "Forma_Golos_Sofridos_Away"]:
    df_full[col] = df_full[col].clip(upper=GOLOS_CAP)


# --- 2. Grupos ---
GROUPS = {
    "A": ["Mexico", "South Africa", "South Korea", "Czech Republic"],
    "B": ["Canada", "Switzerland", "Qatar", "Bosnia and Herzegovina"],
    "C": ["Brazil", "Morocco", "Haiti", "Scotland"],
    "D": ["United States", "Paraguay", "Australia", "Turkey"],
    "E": ["Germany", "Ecuador", "Ivory Coast", "Curaçao"],
    "F": ["Netherlands", "Japan", "Sweden", "Tunisia"],
    "G": ["Belgium", "Egypt", "Iran", "New Zealand"],
    "H": ["Spain", "Uruguay", "Saudi Arabia", "Cape Verde"],
    "I": ["France", "Senegal", "Norway", "Iraq"],
    "J": ["Argentina", "Algeria", "Austria", "Jordan"],
    "K": ["Portugal", "Colombia", "DR Congo", "Uzbekistan"],
    "L": ["England", "Croatia", "Ghana", "Panama"],
}

# Mapa equipa → grupo
TEAM_GROUP = {team: grp for grp, teams in GROUPS.items() for team in teams}

# --- 3. Ranking FIFA para desempate ---
df_ranking = pd.read_csv("../data/fifa_ranking.csv")
NAME_MAP = {
    "south korea": "korea republic", "north korea": "dpr korea",
    "iran": "ir iran", "china": "china pr", "dr congo": "congo dr",
    "czech republic": "czechia", "gambia": "the gambia",
    "kyrgyzstan": "kyrgyz republic", "cape verde": "cabo verde",
    "ivory coast": "côte d'ivoire", "turkey": "türkiye",
    "united states": "usa", "taiwan": "chinese taipei",
    "saint kitts and nevis": "st. kitts and nevis",
    "saint lucia": "st. lucia",
    "saint vincent and the grenadines": "st. vincent / grenadines",
    "united states virgin islands": "us virgin islands",
}
df_ranking["name_norm"] = df_ranking["country_name"].str.strip().str.lower()
rank_lookup = df_ranking.set_index("name_norm")["ranking"]

def get_rank(team):
    key = NAME_MAP.get(team.lower(), team.lower())
    return rank_lookup.get(key, 999)

# --- 4. Inicializar tabela de pontos ---
def init_table(groups):
    records = []
    for grp, teams in groups.items():
        for team in teams:
            records.append({
                "group": grp, "team": team,
                "pts": 0, "gf": 0, "ga": 0, "gd": 0,
                "ranking": get_rank(team)
            })
    return pd.DataFrame(records)

table = init_table(GROUPS)

def update_table(table, home, away, result):
    """Actualiza pontos com base no resultado previsto (H/D/A)."""
    if result == "H":
        table.loc[table["team"] == home, "pts"] += 3
    elif result == "D":
        table.loc[table["team"] == home, "pts"] += 1
        table.loc[table["team"] == away, "pts"] += 1
    else:
        table.loc[table["team"] == away, "pts"] += 3
    return table

 # Após o cap, antes do loop de previsão
# Imputar NaN nas features com a mediana do df_full (jogos históricos)
FEATURES_NUM = [
    "Diff_Ranking", "Fator_Casa",
    "Forma_Golos_Home", "Forma_Pts_Home", "Forma_Golos_Sofridos_Home",
    "Forma_Golos_Away", "Forma_Pts_Away", "Forma_Golos_Sofridos_Away",
]

medians = df_full[FEATURES_NUM].median()

for col in FEATURES_NUM:
    df_full[col] = df_full[col].fillna(medians[col])

print("NaN após imputação:")
print(df_full[FEATURES_NUM].isna().sum())   

# --- 5. Simular fase de grupos ---
wc_2026 = df_full[
    (df_full["tournament"] == "FIFA World Cup") &
    (df_full["date"] >= "2026-06-11")
].copy()

# Jogos já disputados
played = wc_2026[wc_2026["home_score"].notna()].copy()
# Jogos a prever
future = wc_2026[wc_2026["home_score"].isna()].copy()

print(f"Jogos já disputados: {len(played)}")
print(f"Jogos a prever:      {len(future)}")

# Processar jogos já disputados
for _, row in played.iterrows():
    if row["home_score"] > row["away_score"]:
        result = "H"
    elif row["home_score"] == row["away_score"]:
        result = "D"
    else:
        result = "A"
    table = update_table(table, row["home_team"], row["away_team"], result)

# Processar jogos futuros com o modelo
future["Fator_Casa"]      = 0
future["Tipo_Competicao"] = "Mundial"

predicted_results = []
for _, row in future.iterrows():
    X = pd.DataFrame([row[FEATURES]])
    result = clf.predict(X)[0]
    predicted_results.append({
        "home_team": row["home_team"],
        "away_team": row["away_team"],
        "predicted": result
    })
    table = update_table(table, row["home_team"], row["away_team"], result)

df_predicted = pd.DataFrame(predicted_results)

# --- 6. Classificação final dos grupos ---
table_sorted = (
    table
    .sort_values(["group", "pts", "ranking"], ascending=[True, False, True])
    .reset_index(drop=True)
)
table_sorted["position"] = table_sorted.groupby("group").cumcount() + 1

print("\n=== CLASSIFICAÇÃO FINAL DOS GRUPOS ===")
for grp in sorted(GROUPS.keys()):
    grp_df = table_sorted[table_sorted["group"] == grp][["position","team","pts","ranking"]]
    print(f"\nGrupo {grp}:")
    print(grp_df.to_string(index=False))

NaN após imputação:
Diff_Ranking                 0
Fator_Casa                   0
Forma_Golos_Home             0
Forma_Pts_Home               0
Forma_Golos_Sofridos_Home    0
Forma_Golos_Away             0
Forma_Pts_Away               0
Forma_Golos_Sofridos_Away    0
dtype: int64
Jogos já disputados: 28
Jogos a prever:      44

=== CLASSIFICAÇÃO FINAL DOS GRUPOS ===

Grupo A:
 position           team  pts  ranking
        1         Mexico    9       11
        2    South Korea    6       23
        3 Czech Republic    1       44
        4   South Africa    1       61

Grupo B:
 position                   team  pts  ranking
        1            Switzerland    7       19
        2                 Canada    4       27
        3                  Qatar    4       57
        4 Bosnia and Herzegovina    1       64

Grupo C:
 position     team  pts  ranking
        1   Brazil    7        5
        2  Morocco    7        6
        3 Scotland    3       41
        4    Haiti    0       87

Grupo

In [27]:
# --- 7. Extrair qualificados ---

# 1ºs e 2ºs de cada grupo
qualified = {}
thirds = []

for grp in sorted(GROUPS.keys()):
    grp_df = table_sorted[table_sorted["group"] == grp].sort_values(
        ["pts", "ranking"], ascending=[False, True]
    )
    qualified[f"1{grp}"] = grp_df.iloc[0]["team"]
    qualified[f"2{grp}"] = grp_df.iloc[1]["team"]
    thirds.append({
        "group": grp,
        "team":  grp_df.iloc[2]["team"],
        "pts":   grp_df.iloc[2]["pts"],
        "ranking": grp_df.iloc[2]["ranking"],
    })

# 8 melhores terceiros
df_thirds = (
    pd.DataFrame(thirds)
    .sort_values(["pts", "ranking"], ascending=[False, True])
    .reset_index(drop=True)
)
print("=== TERCEIROS CLASSIFICADOS (ordenados) ===")
print(df_thirds.to_string(index=False))

best8_thirds = df_thirds.head(8)
worst4_thirds = df_thirds.iloc[8:]
print(f"\nElimsinados (4 piores terceiros):")
print(worst4_thirds[["group","team","pts"]].to_string(index=False))

# Função para atribuir terceiro a um slot
# Pool = lista de grupos de onde pode vir o terceiro
def assign_third(pool, used_thirds, best8):
    candidates = best8[best8["group"].isin(pool) & ~best8["team"].isin(used_thirds)]
    if candidates.empty:
        # fallback: qualquer terceiro disponível
        candidates = best8[~best8["team"].isin(used_thirds)]
    return candidates.iloc[0]["team"]

used_thirds = []

def get_third(pool):
    team = assign_third(pool, used_thirds, best8_thirds)
    used_thirds.append(team)
    return team

# --- 8. Bracket fase de 32 (jogos 73-88) ---
bracket_32 = {
    73: (qualified["2A"], qualified["2B"]),
    74: (qualified["1E"], get_third(["A","B","C","D","F"])),
    75: (qualified["1F"], qualified["2C"]),
    76: (qualified["1C"], qualified["2F"]),
    77: (qualified["1I"], get_third(["C","D","F","G","H"])),
    78: (qualified["2E"], qualified["2I"]),
    79: (qualified["1A"], get_third(["C","E","F","H","I"])),
    80: (qualified["1L"], get_third(["E","H","I","J","K"])),
    81: (qualified["1D"], get_third(["B","E","F","I","J"])),
    82: (qualified["1G"], get_third(["A","E","H","I","J"])),
    83: (qualified["2K"], qualified["2L"]),
    84: (qualified["1H"], qualified["2J"]),
    85: (qualified["1B"], get_third(["E","F","G","I","J"])),
    86: (qualified["1J"], qualified["2H"]),
    87: (qualified["1K"], get_third(["D","E","I","J","L"])),
    88: (qualified["2D"], qualified["2G"]),
}

print("\n=== FASE DE 32 ===")
for jogo, (home, away) in bracket_32.items():
    print(f"  Jogo {jogo}: {home} vs {away}")

=== TERCEIROS CLASSIFICADOS (ordenados) ===
group           team  pts  ranking
    G          Egypt    4       28
    K       DR Congo    4       43
    B          Qatar    4       57
    H     Cape Verde    4       63
    I         Norway    3       26
    J        Algeria    3       29
    E        Ecuador    3       30
    D         Turkey    3       32
    F         Sweden    3       36
    C       Scotland    3       41
    L          Ghana    3       65
    A Czech Republic    1       44

Elimsinados (4 piores terceiros):
group           team  pts
    F         Sweden    3
    C       Scotland    3
    L          Ghana    3
    A Czech Republic    1

=== FASE DE 32 ===
  Jogo 73: South Korea vs Canada
  Jogo 74: Germany vs Qatar
  Jogo 75: Netherlands vs Morocco
  Jogo 76: Brazil vs Japan
  Jogo 77: France vs Egypt
  Jogo 78: Ivory Coast vs Senegal
  Jogo 79: Mexico vs Cape Verde
  Jogo 80: England vs DR Congo
  Jogo 81: United States vs Norway
  Jogo 82: Belgium vs Algeria
  Jog

In [29]:
# --- 9. Função de simulação de jogo mata-mata ---
def simulate_ko(home, away):
    matches = df_full[
        ((df_full["home_team"] == home) & (df_full["away_team"] == away)) |
        ((df_full["home_team"] == away) & (df_full["away_team"] == home))
    ]

    def get_val(col, default=1.4):
        if len(matches) == 0:
            return default
        val = matches[col].iloc[-1]
        return float(val) if pd.notna(val) else default

    X = pd.DataFrame([{
        "Diff_Ranking":              get_rank(home) - get_rank(away),
        "Fator_Casa":                0,
        "Forma_Golos_Home":          get_val("Forma_Golos_Home"),
        "Forma_Pts_Home":            get_val("Forma_Pts_Home"),
        "Forma_Golos_Sofridos_Home": get_val("Forma_Golos_Sofridos_Home"),
        "Forma_Golos_Away":          get_val("Forma_Golos_Away"),
        "Forma_Pts_Away":            get_val("Forma_Pts_Away"),
        "Forma_Golos_Sofridos_Away": get_val("Forma_Golos_Sofridos_Away"),
        "Tipo_Competicao":           "Mundial",
    }])

    classes = list(clf.classes_)
    proba   = clf.predict_proba(X)[0]
    p_home  = proba[classes.index("H")]
    p_away  = proba[classes.index("A")]
    total   = p_home + p_away
    p_home_norm = p_home / total
    winner  = home if p_home_norm >= 0.5 else away
    return winner, round(p_home_norm, 3), round(1 - p_home_norm, 3)

# --- 10. Simular todas as fases ---
def simulate_round(matchups, round_name):
    print(f"\n=== {round_name} ===")
    winners = {}
    losers  = {}
    for jogo, (home, away) in matchups.items():
        winner, ph, pa = simulate_ko(home, away)
        loser = away if winner == home else home
        winners[jogo] = winner
        losers[jogo]  = loser
        print(f"  Jogo {jogo}: {home} vs {away} → {winner} ({ph} vs {pa})")
    return winners, losers

# Fase de 32
w32, l32 = simulate_round(bracket_32, "FASE DE 32")

# Oitavos
bracket_16 = {
    89: (w32[74], w32[77]),
    90: (w32[73], w32[75]),
    91: (w32[76], w32[78]),
    92: (w32[79], w32[80]),
    93: (w32[83], w32[84]),
    94: (w32[81], w32[82]),
    95: (w32[86], w32[88]),
    96: (w32[85], w32[87]),
}
w16, _ = simulate_round(bracket_16, "OITAVOS DE FINAL")

# Quartos
bracket_qf = {
    97:  (w16[89], w16[90]),
    98:  (w16[93], w16[94]),
    99:  (w16[91], w16[92]),
    100: (w16[95], w16[96]),
}
wqf, _ = simulate_round(bracket_qf, "QUARTOS DE FINAL")

# Meias
bracket_sf = {
    101: (wqf[97],  wqf[98]),
    102: (wqf[99],  wqf[100]),
}
wsf, lsf = simulate_round(bracket_sf, "MEIAS-FINAIS")

# 3º lugar
bracket_3rd = {103: (lsf[101], lsf[102])}
w3rd, _ = simulate_round(bracket_3rd, "DISPUTA DO 3º LUGAR")

# Final
bracket_final = {104: (wsf[101], wsf[102])}
wfinal, _ = simulate_round(bracket_final, "FINAL")

print(f"\n{'='*50}")
print(f"🏆 CAMPEÃO DO MUNDO 2026: {wfinal[104]}")
print(f"🥈 Vice-campeão:          {[t for t in [wsf[101],wsf[102]] if t != wfinal[104]][0]}")
print(f"🥉 3º lugar:              {w3rd[103]}")


=== FASE DE 32 ===
  Jogo 73: South Korea vs Canada → South Korea (0.6 vs 0.4)
  Jogo 74: Germany vs Qatar → Germany (0.87 vs 0.13)
  Jogo 75: Netherlands vs Morocco → Netherlands (0.715 vs 0.285)
  Jogo 76: Brazil vs Japan → Brazil (0.729 vs 0.271)
  Jogo 77: France vs Egypt → France (0.916 vs 0.084)
  Jogo 78: Ivory Coast vs Senegal → Ivory Coast (0.557 vs 0.443)
  Jogo 79: Mexico vs Cape Verde → Mexico (0.847 vs 0.153)
  Jogo 80: England vs DR Congo → England (0.895 vs 0.105)
  Jogo 81: United States vs Norway → United States (0.777 vs 0.223)
  Jogo 82: Belgium vs Algeria → Belgium (0.953 vs 0.047)
  Jogo 83: Portugal vs Croatia → Croatia (0.342 vs 0.658)
  Jogo 84: Spain vs Austria → Austria (0.451 vs 0.549)
  Jogo 85: Switzerland vs Ecuador → Switzerland (0.875 vs 0.125)
  Jogo 86: Argentina vs Uruguay → Argentina (0.543 vs 0.457)
  Jogo 87: Colombia vs Turkey → Turkey (0.45 vs 0.55)
  Jogo 88: Australia vs Iran → Australia (0.538 vs 0.462)

=== OITAVOS DE FINAL ===
  Jogo 89: Ge

O que mudou no V3:

França chega à final (era Holanda no V2, Bélgica no V1)
Croácia elimina Portugal (83: 0.658) — o modelo penaliza Portugal por ter ficado em 2º no grupo K atrás da Colômbia
Áustria elimina Espanha novamente — padrão consistente entre versões
Final França vs México → 0.189 vs 0.811 — o modelo tem convicção no México

A persistência do México em todas as versões diz algo importante:

O bracket favorece o México estruturalmente — lado da tabela sem Argentina, França ou Alemanha até à final. Não é uma anomalia do modelo, é uma realidade do sorteio. O modelo está a ser consistente.
Nota curiosa — Jogo 104:

França vs México com 0.811 para o México é a previsão mais confiante de toda a simulação. O Diff_Ranking do México vs França (11 vs 2) deveria favorecer a França — mas as features de forma inflacionadas do México dominam. É exactamente a Limitação 1 documentada